<a href="https://colab.research.google.com/github/zzbchs/Restaurant-reviews/blob/main/restaurant_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Project Context
Customer reviews are critical for restaurant growth and improving overall customer satisfaction. We have collected a dataset containing 1,000 customer reviews.

Dataset Overview
Columns:

Review: Text feedback submitted by the customer.

Liked (or Like): Binary label where 1 indicates a positive review and 0 indicates a negative review.

Your Tasks & Deliverables
Data Preprocessing & EDA:

Clean the text data (handle lowercasing, stop words, punctuation, and stemming/lemmatization).

Perform basic Exploratory Data Analysis (e.g., review lengths, most common positive/negative words, class distribution).

Feature Engineering:

Convert text data into numerical features using techniques like Bag of Words (CountVectorizer) or TF-IDF.

Model Building & Evaluation:

Train at least two classification algorithms (e.g., Naive Bayes, Logistic Regression, Random Forest).

Evaluate model performance using metrics such as Accuracy, Precision, Recall, and F1-Score.

Compare the models and identify which one performs best.

Project Output:

Push your clean, well-commented code to a GitHub repository (including a README.md explaining your approach and findings).

Prepare a brief summary of your results and any insights you discovered."""

In [21]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Download NLTK resources (if not already downloaded)
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [13]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [22]:
df = pd.read_csv('Restaurant_Reviews.csv', sep='\t')
display(df.head())

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [5]:
df.shape

(1000, 2)

**Data Preprocessing & EDA:**

Clean the text data (handle lowercasing, stop words, punctuation, and stemming/lemmatization).

Perform basic Exploratory Data Analysis (e.g., review lengths, most common positive/negative words, class distribution).



In [ ]:
#Clean the text data (handle lowercasing, stop words, punctuation, and stemming/lemmatization).

In [23]:
#convert everything in the column review to lowercase as it doesn't add any material value
df['Review'] = df['Review'].str.lower()
# Remove punctuation
df['Review'] = df['Review'].apply(lambda x: re.sub(r'[^"\w\s"]', '', x))

# Remove stop words
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    return ' '.join([word for word in str(text).split() if word not in stop_words])
df['Review'] = df['Review'].apply(remove_stopwords)
display(df.head())

,Review,Liked
0,wow loved place,1
1,crust good,0
2,tasty texture nasty,0
3,stopped late may bank holiday rick steve recom...,1
4,selection menu great prices,1


### Stemming

Stemming is a process of reducing inflected (or sometimes derived) words to their word stem, base or root form—generally a written word form. The stem is not necessarily identical to the morphological root of the word; it is usually sufficient that related words map to the same stem, even if the stem is not a valid root itself. For example, 'consultant', 'consulting', 'consultative' could all be reduced to the stem 'consult'.

In [19]:
"""ps = PorterStemmer()

def stem_text(text):
    # Tokenize the text first
    words = nltk.word_tokenize(text)
    # Apply stemming to each word
    stemmed_words = [ps.stem(word) for word in words]
    # Join the stemmed words back into a string
    return ' '.join(stemmed_words)

df['Review'] = df['Review'].apply(stem_text)
display(df.head())"""

"ps = PorterStemmer()\n\ndef stem_text(text):\n    # Tokenize the text first\n    words = nltk.word_tokenize(text)\n    # Apply stemming to each word\n    stemmed_words = [ps.stem(word) for word in words]\n    # Join the stemmed words back into a string\n    return ' '.join(stemmed_words)\n\ndf['Review'] = df['Review'].apply(stem_text)\ndisplay(df.head())"

In [24]:
lemmatizer = nltk.WordNetLemmatizer()

def lemmatize_text(text):
    # Tokenize the text first
    words = nltk.word_tokenize(text)
    # Apply lemmatization to each word
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    # Join the lemmatized words back into a string
    return ' '.join(lemmatized_words)

df['Review'] = df['Review'].apply(lemmatize_text)
display(df.head())

,Review,Liked
0,wow loved place,1
1,crust good,0
2,tasty texture nasty,0
3,stopped late may bank holiday rick steve recom...,1
4,selection menu great price,1


### Lemmatization

Lemmatization, unlike stemming, reduces words to their meaningful base form, called a lemma, which is a dictionary word. It takes into account the morphological analysis of words, usually requiring part-of-speech (POS) tags to be most effective. This often results in a more semantically correct base word than stemming. For instance, 'better' would be lemmatized to 'good', whereas a stemmer might just leave it as 'better' or 'bet'.



**Feature Engineering:**

Convert text data into numerical features using techniques like Bag of Words (CountVectorizer) or TF-IDF.

In [39]:
# Bag of Words (CountVectorizer)
print("Applying Bag of Words (CountVectorizer):")
cv = CountVectorizer(max_features=1500) # You can adjust max_features as needed
X_bow = cv.fit_transform(df['Review']).toarray()
y_bow = df['Liked'].values

print(f"Shape of Bag of Words matrix: {X_bow.shape}")
print(f"Sample features: {cv.get_feature_names_out()[:1000]}\n")

# TF-IDF Vectorizer
print("Applying TF-IDF Vectorizer:")
tfidf = TfidfVectorizer(max_features=1500) # You can adjust max_features as needed
X_tfidf = tfidf.fit_transform(df['Review']).toarray()
y_tfidf = df['Liked'].values

print(f"Shape of TF-IDF matrix: {X_tfidf.shape}")
print(f"Sample features: {tfidf.get_feature_names_out()[:1000]}")

Applying Bag of Words (CountVectorizer):
Shape of Bag of Words matrix: (1000, 1500)
Sample features: ['10' '100' '12' '1979' '20' '2007' '23' '30' '34ths' '35' '40' '400'
 '40min' '45' '70' '785' '90' 'absolute' 'absolutely' 'acknowledged'
 'actually' 'added' 'ago' 'almost' 'also' 'although' 'always' 'amazing'
 'ambiance' 'ambience' 'amount' 'another' 'anyone' 'anything' 'anytime'
 'anyway' 'appetizer' 'area' 'arent' 'around' 'arrived' 'ask' 'asked'
 'assure' 'ate' 'atmosphere' 'attack' 'attentive' 'attitude' 'authentic'
 'average' 'avoid' 'away' 'awesome' 'awful' 'baby' 'bachi' 'back' 'bacon'
 'bad' 'bagel' 'bakery' 'bar' 'barely' 'bartender' 'basically' 'bathroom'
 'batter' 'bay' 'bean' 'beat' 'beautiful' 'become' 'beef' 'beer' 'behind'
 'believe' 'belly' 'best' 'better' 'beyond' 'big' 'bill' 'biscuit'
 'bisque' 'bit' 'bite' 'black' 'bland' 'blow' 'boba' 'boot' 'bother'
 'bowl' 'box' 'boy' 'boyfriend' 'bread' 'break' 'breakfast' 'brick'
 'bring' 'brought' 'brunch' 'buck' 'buffet' 'bu

In [ ]:
"""Model Building & Evaluation:

Train at least two classification algorithms (e.g., Naive Bayes, Logistic Regression, Random Forest).

Evaluate model performance using metrics such as Accuracy, Precision, Recall, and F1-Score.

Compare the models and identify which one performs best."""


## Model Building and Evaluation with Bag of Words (BoW) Features

In [31]:
# Split data into training and testing sets for Bag of Words
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(X_bow, y_bow, test_size=0.2, random_state=42)

print(f"X_train_bow shape: {X_train_bow.shape}")
print(f"X_test_bow shape: {X_test_bow.shape}")
print(f"y_train_bow shape: {y_train_bow.shape}")
print(f"y_test_bow shape: {y_test_bow.shape}")

X_train_bow shape: (800, 1500)
X_test_bow shape: (200, 1500)
y_train_bow shape: (800,)
y_test_bow shape: (200,)


### Multinomial Naive Bayes with BoW

In [32]:
# Train Multinomial Naive Bayes model
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train_bow)

# Predict on test data
y_pred_mnb_bow = mnb_bow.predict(X_test_bow)

# Evaluate the model
print("Multinomial Naive Bayes (BoW) Performance:")
print(f"Accuracy: {accuracy_score(y_test_bow, y_pred_mnb_bow):.4f}")
print(f"Precision: {precision_score(y_test_bow, y_pred_mnb_bow):.4f}")
print(f"Recall: {recall_score(y_test_bow, y_pred_mnb_bow):.4f}")
print(f"F1-Score: {f1_score(y_test_bow, y_pred_mnb_bow):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_bow, y_pred_mnb_bow))

Multinomial Naive Bayes (BoW) Performance:
Accuracy: 0.7500
Precision: 0.7596
Recall: 0.7596
F1-Score: 0.7596
Confusion Matrix:
 [[71 25]
 [25 79]]


### Logistic Regression with BoW

In [33]:
# Train Logistic Regression model
log_reg_bow = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' for small datasets and L1/L2 regularization
log_reg_bow.fit(X_train_bow, y_train_bow)

# Predict on test data
y_pred_log_reg_bow = log_reg_bow.predict(X_test_bow)

# Evaluate the model
print("Logistic Regression (BoW) Performance:")
print(f"Accuracy: {accuracy_score(y_test_bow, y_pred_log_reg_bow):.4f}")
print(f"Precision: {precision_score(y_test_bow, y_pred_log_reg_bow):.4f}")
print(f"Recall: {recall_score(y_test_bow, y_pred_log_reg_bow):.4f}")
print(f"F1-Score: {f1_score(y_test_bow, y_pred_log_reg_bow):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_bow, y_pred_log_reg_bow))

Logistic Regression (BoW) Performance:
Accuracy: 0.7750
Precision: 0.8242
Recall: 0.7212
F1-Score: 0.7692
Confusion Matrix:
 [[80 16]
 [29 75]]


### Random Forest with BoW

In [34]:
# Train Random Forest Classifier model
rf_bow = RandomForestClassifier(n_estimators=100, random_state=42)
rf_bow.fit(X_train_bow, y_train_bow)

# Predict on test data
y_pred_rf_bow = rf_bow.predict(X_test_bow)

# Evaluate the model
print("Random Forest (BoW) Performance:")
print(f"Accuracy: {accuracy_score(y_test_bow, y_pred_rf_bow):.4f}")
print(f"Precision: {precision_score(y_test_bow, y_pred_rf_bow):.4f}")
print(f"Recall: {recall_score(y_test_bow, y_pred_rf_bow):.4f}")
print(f"F1-Score: {f1_score(y_test_bow, y_pred_rf_bow):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_bow, y_pred_rf_bow))

Random Forest (BoW) Performance:
Accuracy: 0.7400
Precision: 0.8171
Recall: 0.6442
F1-Score: 0.7204
Confusion Matrix:
 [[81 15]
 [37 67]]


## Model Building and Evaluation with TF-IDF Features

In [35]:
# Split data into training and testing sets for TF-IDF
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(X_tfidf, y_tfidf, test_size=0.2, random_state=42)

print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"X_test_tfidf shape: {X_test_tfidf.shape}")
print(f"y_train_tfidf shape: {y_train_tfidf.shape}")
print(f"y_test_tfidf shape: {y_test_tfidf.shape}")

X_train_tfidf shape: (800, 1500)
X_test_tfidf shape: (200, 1500)
y_train_tfidf shape: (800,)
y_test_tfidf shape: (200,)


### Multinomial Naive Bayes with TF-IDF

In [36]:
# Train Multinomial Naive Bayes model
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train_tfidf)

# Predict on test data
y_pred_mnb_tfidf = mnb_tfidf.predict(X_test_tfidf)

# Evaluate the model
print("Multinomial Naive Bayes (TF-IDF) Performance:")
print(f"Accuracy: {accuracy_score(y_test_tfidf, y_pred_mnb_tfidf):.4f}")
print(f"Precision: {precision_score(y_test_tfidf, y_pred_mnb_tfidf):.4f}")
print(f"Recall: {recall_score(y_test_tfidf, y_pred_mnb_tfidf):.4f}")
print(f"F1-Score: {f1_score(y_test_tfidf, y_pred_mnb_tfidf):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_tfidf, y_pred_mnb_tfidf))

Multinomial Naive Bayes (TF-IDF) Performance:
Accuracy: 0.7900
Precision: 0.8100
Recall: 0.7788
F1-Score: 0.7941
Confusion Matrix:
 [[77 19]
 [23 81]]


### Logistic Regression with TF-IDF

In [37]:
# Train Logistic Regression model
log_reg_tfidf = LogisticRegression(random_state=42, solver='liblinear')
log_reg_tfidf.fit(X_train_tfidf, y_train_tfidf)

# Predict on test data
y_pred_log_reg_tfidf = log_reg_tfidf.predict(X_test_tfidf)

# Evaluate the model
print("Logistic Regression (TF-IDF) Performance:")
print(f"Accuracy: {accuracy_score(y_test_tfidf, y_pred_log_reg_tfidf):.4f}")
print(f"Precision: {precision_score(y_test_tfidf, y_pred_log_reg_tfidf):.4f}")
print(f"Recall: {recall_score(y_test_tfidf, y_pred_log_reg_tfidf):.4f}")
print(f"F1-Score: {f1_score(y_test_tfidf, y_pred_log_reg_tfidf):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_tfidf, y_pred_log_reg_tfidf))

Logistic Regression (TF-IDF) Performance:
Accuracy: 0.7700
Precision: 0.8372
Recall: 0.6923
F1-Score: 0.7579
Confusion Matrix:
 [[82 14]
 [32 72]]


### Random Forest with TF-IDF

In [38]:
# Train Random Forest Classifier model
rf_tfidf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_tfidf.fit(X_train_tfidf, y_train_tfidf)

# Predict on test data
y_pred_rf_tfidf = rf_tfidf.predict(X_test_tfidf)

# Evaluate the model
print("Random Forest (TF-IDF) Performance:")
print(f"Accuracy: {accuracy_score(y_test_tfidf, y_pred_rf_tfidf):.4f}")
print(f"Precision: {precision_score(y_test_tfidf, y_pred_rf_tfidf):.4f}")
print(f"Recall: {recall_score(y_test_tfidf, y_pred_rf_tfidf):.4f}")
print(f"F1-Score: {f1_score(y_test_tfidf, y_pred_rf_tfidf):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_tfidf, y_pred_rf_tfidf))

Random Forest (TF-IDF) Performance:
Accuracy: 0.7450
Precision: 0.8193
Recall: 0.6538
F1-Score: 0.7273
Confusion Matrix:
 [[81 15]
 [36 68]]


## Model Comparison and Best Model Identification

Based on the evaluation metrics above, we can compare the performance of the different models using both Bag of Words (BoW) and TF-IDF features. The goal is to identify which combination yields the best results for our text classification task.

### Summary of Results:

| Model                  | Feature Set | Accuracy | Precision | Recall | F1-Score |
| :--------------------- | :---------- | :------- | :-------- | :----- | :------- |
| Multinomial Naive Bayes| BoW         | 0.7500   | 0.7596    | 0.7596 | 0.7596   |
| Logistic Regression    | BoW         | 0.7750   | 0.8242    | 0.7212 | 0.7692   |
| Random Forest          | BoW         | 0.7400   | 0.8171    | 0.6442 | 0.7204   |
| Multinomial Naive Bayes| TF-IDF      | 0.7900   | 0.8100    | 0.7788 | 0.7941   |
| Logistic Regression    | TF-IDF      | 0.7700   | 0.8372    | 0.6923 | 0.7579   |
| Random Forest          | TF-IDF      | 0.7450   | 0.8193    | 0.6538 | 0.7273   |